In [4]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import time
import re

# 定数
DB_NAME = "google_repos.db"
TARGET_URL = "https://github.com/google?tab=repositories"

def setup_database():
    """データベースとテーブルを作成する関数"""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    # テーブルが存在しない場合のみ作成
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS repositories (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            language TEXT,
            stars TEXT
        )
    ''')
    
    # 既存のデータをクリア（再実行用）
    cursor.execute('DELETE FROM repositories')
    
    conn.commit()
    conn.close()
    print(f"[INFO] データベース {DB_NAME} を初期化しました。")

def get_google_repos():
    """GithubのGoogleページからリポジトリ情報をスクレイピングする関数"""
    print(f"[INFO] {TARGET_URL} からデータを取得中...")
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    
    try:
        response = requests.get(TARGET_URL, headers=headers)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"[ERROR] ページの取得に失敗しました: {e}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')
    
    # リポジトリリストのコンテナを取得（GithubのHTML構造に基づく）
    repo_list = soup.find_all('li', itemprop='owns')
    
    data_list = []
    
    for repo in repo_list:
        # 1. リポジトリ名
        name_tag = repo.find('a', itemprop='name codeRepository')
        name = name_tag.get_text(strip=True) if name_tag else "Unknown"
        
        # 2. 主要言語
        lang_tag = repo.find('span', itemprop='programmingLanguage')
        language = lang_tag.get_text(strip=True) if lang_tag else "No Language"
        
        # 3. スターの数
        # スターへのリンクは通常 /stargazers で終わるhref属性を持つ
        star_tag = repo.find('a', href=re.compile(r'/stargazers$'))
        # アイコンのテキストではなく、数字部分を取得するために整形
        stars = star_tag.get_text(strip=True) if star_tag else "0"
        
        repo_data = (name, language, stars)
        data_list.append(repo_data)
        
        print(f"[SCRAPED] {name} | {language} | ⭐ {stars}")
        
        # 要件: ちゃんとtime.sleep(1)を入れる
        time.sleep(1)
        
    return data_list

def save_to_db(data_list):
    """取得したデータをデータベースに保存する関数"""
    if not data_list:
        print("[WARN] 保存するデータがありません。")
        return

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    cursor.executemany('INSERT INTO repositories (name, language, stars) VALUES (?, ?, ?)', data_list)
    
    conn.commit()
    conn.close()
    print(f"[INFO] {len(data_list)} 件のデータを保存しました。")

def show_saved_data():
    """保存されたデータをSELECT文で表示する関数"""
    print("\n--- データベース保存結果 (SELECT文による表示) ---")
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    # データを取得
    cursor.execute('SELECT * FROM repositories')
    rows = cursor.fetchall()
    
    # 表示形式を整える
    print(f"{'ID':<5} | {'Name':<30} | {'Language':<15} | {'Stars':<10}")
    print("-" * 70)
    for row in rows:
        # rowは (id, name, language, stars)
        print(f"{row[0]:<5} | {row[1]:<30} | {row[2]:<15} | {row[3]:<10}")
        
    conn.close()

if __name__ == "__main__":
    # 1. DB作成
    setup_database()
    
    # 2. スクレイピング
    scraped_data = get_google_repos()
    
    # 3. DB保存
    save_to_db(scraped_data)
    
    # 4. 結果表示
    show_saved_data()

[INFO] データベース google_repos.db を初期化しました。
[INFO] https://github.com/google?tab=repositories からデータを取得中...
[WARN] 保存するデータがありません。

--- データベース保存結果 (SELECT文による表示) ---
ID    | Name                           | Language        | Stars     
----------------------------------------------------------------------


In [6]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import time
import re

# 定数
DB_NAME = "google_repos.db"
TARGET_URL = "https://github.com/google?tab=repositories"

def setup_database():
    """データベースとテーブルを作成する関数"""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS repositories (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            language TEXT,
            stars TEXT
        )
    ''')
    
    cursor.execute('DELETE FROM repositories')
    conn.commit()
    conn.close()
    print(f"[INFO] データベース {DB_NAME} を初期化しました。")

def get_google_repos():
    """GithubのGoogleページからリポジトリ情報をスクレイピングする関数"""
    print(f"[INFO] {TARGET_URL} からデータを取得中...")
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    
    try:
        response = requests.get(TARGET_URL, headers=headers)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"[ERROR] ページの取得に失敗しました: {e}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')

    # 【デバッグ用】ページタイトルを表示（もしLogin画面ならブロックされています）
    if soup.title:
        print(f"[PAGE TITLE] {soup.title.get_text(strip=True)}")

    # 【修正点】特定のIDやクラスに頼らず、すべての li タグから「リポジトリ」を探す戦略
    all_lis = soup.find_all('li')
    
    data_list = []
    found_count = 0
    
    print("[INFO] ページ内のリスト要素を解析中...")

    for li in all_lis:
        # 1. リポジトリ名の特定
        # Githubのリポジトリ名は通常 h3 タグの中の a タグにある
        h3 = li.find('h3')
        if not h3:
            continue
            
        a_tag = h3.find('a')
        if not a_tag:
            continue
            
        # リンクのhrefが /google/ で始まっているものをリポジトリとみなす
        href = a_tag.get('href', '')
        if not href.startswith('/google/'):
            continue
            
        # ここまで条件が揃えば、これはリポジトリのデータです
        name = a_tag.get_text(strip=True)
        
        # 2. 主要言語の特定 (itemprop="programmingLanguage")
        lang_tag = li.find('span', itemprop='programmingLanguage')
        language = lang_tag.get_text(strip=True) if lang_tag else "No Language"
        
        # 3. スター数の特定 (hrefが /stargazers で終わるリンク)
        star_tag = li.find('a', href=re.compile(r'/stargazers$'))
        if star_tag:
            stars = star_tag.get_text(strip=True)
        else:
            stars = "0"
        
        repo_data = (name, language, stars)
        data_list.append(repo_data)
        found_count += 1
        
        print(f"[SCRAPED] {name} | {language} | ⭐ {stars}")
        
        # time.sleepを入れる
        time.sleep(1)
        
        # 最初の30件取得したら終了（無限ループ防止や負荷軽減のため）
        if found_count >= 30:
            break
            
    if found_count == 0:
        print("[ERROR] データが見つかりませんでした。Github側の仕様変更か、Bot検知された可能性があります。")
        
    return data_list

def save_to_db(data_list):
    """取得したデータをデータベースに保存する関数"""
    if not data_list:
        print("[WARN] 保存するデータがありません。")
        return

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.executemany('INSERT INTO repositories (name, language, stars) VALUES (?, ?, ?)', data_list)
    conn.commit()
    conn.close()
    print(f"[INFO] {len(data_list)} 件のデータを保存しました。")

def show_saved_data():
    """保存されたデータを表示する関数"""
    print("\n--- データベース保存結果 (SELECT文による表示) ---")
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute('SELECT * FROM repositories')
    rows = cursor.fetchall()
    
    if not rows:
        print("データが存在しません。")
    else:
        print(f"{'ID':<5} | {'Name':<35} | {'Language':<15} | {'Stars':<10}")
        print("-" * 75)
        for row in rows:
            print(f"{row[0]:<5} | {row[1]:<35} | {row[2]:<15} | {row[3]:<10}")
        
    conn.close()

if __name__ == "__main__":
    setup_database()
    scraped_data = get_google_repos()
    save_to_db(scraped_data)
    show_saved_data()

[INFO] データベース google_repos.db を初期化しました。
[INFO] https://github.com/google?tab=repositories からデータを取得中...
[PAGE TITLE] Google · GitHub
[INFO] ページ内のリスト要素を解析中...
[ERROR] データが見つかりませんでした。Github側の仕様変更か、Bot検知された可能性があります。
[WARN] 保存するデータがありません。

--- データベース保存結果 (SELECT文による表示) ---
データが存在しません。


In [8]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import time
import re

# 定数
DB_NAME = "google_repos.db"
TARGET_URL = "https://github.com/google?tab=repositories"

def setup_database():
    """データベースとテーブルを作成する関数"""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS repositories (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            language TEXT,
            stars TEXT
        )
    ''')
    
    cursor.execute('DELETE FROM repositories')
    conn.commit()
    conn.close()
    print(f"[INFO] データベース {DB_NAME} を初期化しました。")

def get_google_repos():
    """GithubのGoogleページからリポジトリ情報をスクレイピングする関数"""
    print(f"[INFO] {TARGET_URL} からデータを取得中...")
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    
    try:
        response = requests.get(TARGET_URL, headers=headers)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"[ERROR] ページの取得に失敗しました: {e}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')

    # 【デバッグ】タイトル確認
    if soup.title:
        print(f"[PAGE TITLE] {soup.title.get_text(strip=True)}")

    data_list = []
    
    # 【新戦略】リスト構造(li)に依存せず、リポジトリ名である「h3タグ内のリンク」を全検索する
    # Githubのリポジトリ名はほぼ確実に <h3> <a href="/org/repo">name</a> </h3> の構造です
    all_h3 = soup.find_all('h3')
    
    print(f"[DEBUG] ページ内の全h3タグ数: {len(all_h3)}")
    
    for h3 in all_h3:
        a_tag = h3.find('a')
        if not a_tag:
            continue
            
        href = a_tag.get('href', '')
        
        # リンクが '/google/' で始まり、かつ '/google' そのものではないものをリポジトリと判定
        if href.startswith('/google/') and href != '/google':
            name = a_tag.get_text(strip=True)
            
            # このh3タグを含む親コンテナ（liタグなど）を探して、その中にある言語やスターを探す
            # h3 -> div -> div -> li のように階層を遡る
            container = h3.find_parent('li')
            
            # もしliが見つからない場合は、h3の親要素そのものをコンテナとして扱う（念のため）
            if not container:
                container = h3.parent
            
            # 1. 言語 (itemprop="programmingLanguage")
            lang_tag = container.find('span', itemprop='programmingLanguage')
            language = lang_tag.get_text(strip=True) if lang_tag else "No Language"
            
            # 2. スター数 (hrefが /stargazers で終わるリンク)
            star_tag = container.find('a', href=re.compile(r'/stargazers$'))
            if star_tag:
                stars = star_tag.get_text(strip=True).replace(',', '') # 3,000 などのカンマを除去
            else:
                stars = "0"
            
            # データをリストに追加
            repo_data = (name, language, stars)
            data_list.append(repo_data)
            
            print(f"[SCRAPED] {name} | {language} | ⭐ {stars}")
            
            # 要件: time.sleep(1)
            time.sleep(1)
            
            # 30件ほど取れたら十分なのでループを抜ける（サーバー負荷軽減）
            if len(data_list) >= 30:
                break
    
    if not data_list:
        print("[ERROR] h3タグ検索でもデータが見つかりませんでした。")

    return data_list

def save_to_db(data_list):
    if not data_list:
        print("[WARN] 保存するデータがありません。")
        return

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.executemany('INSERT INTO repositories (name, language, stars) VALUES (?, ?, ?)', data_list)
    conn.commit()
    conn.close()
    print(f"[INFO] {len(data_list)} 件のデータを保存しました。")

def show_saved_data():
    print("\n--- データベース保存結果 (SELECT文による表示) ---")
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute('SELECT * FROM repositories')
    rows = cursor.fetchall()
    
    if not rows:
        print("データが存在しません。")
    else:
        print(f"{'ID':<5} | {'Name':<35} | {'Language':<15} | {'Stars':<10}")
        print("-" * 75)
        for row in rows:
            print(f"{row[0]:<5} | {row[1]:<35} | {row[2]:<15} | {row[3]:<10}")
        
    conn.close()

if __name__ == "__main__":
    setup_database()
    scraped_data = get_google_repos()
    save_to_db(scraped_data)
    show_saved_data()

[INFO] データベース google_repos.db を初期化しました。
[INFO] https://github.com/google?tab=repositories からデータを取得中...
[PAGE TITLE] Google · GitHub
[DEBUG] ページ内の全h3タグ数: 3
[ERROR] h3タグ検索でもデータが見つかりませんでした。
[WARN] 保存するデータがありません。

--- データベース保存結果 (SELECT文による表示) ---
データが存在しません。
